In [7]:
import json
import pandas as pd
import re

folder_path = '../data/'
result_path = folder_path + 'results/'

file = result_path + 'CLAUDE_df_evaluated_validation.csv'

In [8]:
df_validation = pd.read_csv(file)
df_validation

,DOI,Title,Authors,Year,Abstract,Include,Reason,Data Source
0,10.1109/ACCESS.2024.3423462,MMW-AQA: Multimodal In-the-Wild Dataset for Ac...,T. Nagai; S. Takeda; S. Suzuki; H. Seshimo,2024,Action quality assessment (AQA) is a task for ...,yes,"The article presents MMW-AQA, a multimodal dat...",MMW-AQA
1,10.23919/EUSIPCO55093.2022.9909680,Clotho-AQA: A Crowdsourced Dataset for Audio Q...,S. Lipping; P. Sudarsanam; K. Drossos; T. Virt...,2022,Audio question answering (AQA) is a multimodal...,no,While the article presents Clotho-AQA dataset ...,NaN
2,10.1109/ICAACE61206.2024.10548995,Action Quality Assessment with Multi-scale Tem...,W. Wang; H. Wang; Y. Hao; Q. Wang,2024,Action quality assessment is a more challengin...,yes,The article proposes an action quality assessm...,AQA-7;MTL-AQA
3,10.1109/ITME53901.2021.00048,Skeleton Based Action Quality Assessment of Fi...,H. -Y. Li; Q. Lei; H. -B. Zhang; J. -X. Du,2021,Action quality assessment(AQA) aims at achievi...,yes,The article proposes a skeleton-based AQA meth...,MIT
4,10.1109/ICIP42928.2021.9506257,Action Quality Assessment With Ignoring Scene ...,T. Nagai; S. Takeda; M. Matsumura; S. Shimizu;...,2021,We propose an action quality assessment (AQA) ...,yes,The article proposes an AQA method evaluated o...,Diving
...,...,...,...,...,...,...,...,...
140,10.1109/TCSVT.2020.3017727,Action Quality Assessment Using Siamese Networ...,"Jain, Hiteshi; Harit, Gaurav; Sharma, Avinash",2021,Automated vision-based score estimation models...,yes,Meets all inclusion criteria: (a) uses video d...,NaN
141,10.1007/s11042-012-1097-x,An audio-visual approach to web video categori...,"Ionescu, Bogdan Emanuel; Seyerlehner, Klaus; M...",2014,"In this paper, we discuss and audio-visual app...",no,Not related to Human Action Quality Assessment...,NaN
142,10.1155/2017/9281321,A Multicriteria Intelligence Aid Methodology U...,"Frini, Anissa",2017,Intelligence is increasingly relevant today in...,no,Not related to Human Action Quality Assessment...,NaN
143,10.1038/s41558-018-0343-2,An assessment of climate action by high-carbon...,"Dietz, Simon; Fruitiere, Charles; Garcia-Manas...",2018,Corporations are an important source of GHG em...,no,Not related to Human Action Quality Assessment...,NaN


In [9]:
with open(folder_path + 'study_datasets.json', 'r') as f:
    study_papers_list = json.load(f)

study_papers_list

{'10.1145/2505323.2505330': 'Perturbed Workout; SU-41 Gesture',
 '10.1109/CVPRW.2013.82': 'Stroke Rehabilitation',
 '10.1109/EMBC.2014.6944292': 'self-created',
 '10.1007/978-3-319-10599-4_36': 'MIT-Dive; MIT-Skate',
 '10.1109/ICCT.2015.7399879': 'self-created',
 '10.1109/JBHI.2014.2375206': 'self-created',
 '10.1109/TPAMI.2016.2533388': 'Stroke Rehabilitation',
 '10.1109/CVPRW.2017.16': 'MIT-Dive; MIT-Skate; UNLV-Dive; UNLV-Vault',
 '10.1007/978-3-030-00767-6_12': 'UNLV-Dive; UNLV-Vault; MIT-Skate',
 '10.1109/ICIP.2018.8451364': 'UNLV-Dive',
 '10.1109/CVPR.2019.00039': 'MTL-AQA',
 '10.1007/978-3-030-20876-9_10': 'MIT-Dive; UNLV-Dive; UNLV-Vault',
 '10.1109/ICCV.2019.00643': 'AQA-7; JIGSAWS',
 '10.1109/WACV.2019.00161': 'AQA-7',
 '10.1016/j.compbiomed.2020.104021': 'self-created',
 '10.1109/TransAI49837.2020.00030': 'UI-PRMD',
 '10.1145/3341105.3374092': 'Origami Video',
 '10.1109/CVPR42600.2020.00986': 'AQA-7; MTL-AQA; JIGSAWS',
 '10.1145/3394171.3413560': 'Rhythmic Gymnastics; MIT-Sk

In [10]:
verified_dois_study = set(study_papers_list.keys())
verified_dois_validation = set(df_validation["DOI"].tolist())

matches = verified_dois_study & verified_dois_validation
num_matches = len(matches)

only_study = verified_dois_study - verified_dois_validation
num_study = len(only_study)

only_validation = verified_dois_validation - verified_dois_study
num_validation = len(only_validation)

print("Perfect Matches:", num_matches)
print("Only in Study:", num_study)
print("Only in Validation:", num_validation)

Perfect Matches: 70
Only in Study: 24
Only in Validation: 75


In [11]:
df_validation = df_validation[df_validation["Include"] != "no"].reset_index(drop=True)
df_validation

,DOI,Title,Authors,Year,Abstract,Include,Reason,Data Source
0,10.1109/ACCESS.2024.3423462,MMW-AQA: Multimodal In-the-Wild Dataset for Ac...,T. Nagai; S. Takeda; S. Suzuki; H. Seshimo,2024,Action quality assessment (AQA) is a task for ...,yes,"The article presents MMW-AQA, a multimodal dat...",MMW-AQA
1,10.1109/ICAACE61206.2024.10548995,Action Quality Assessment with Multi-scale Tem...,W. Wang; H. Wang; Y. Hao; Q. Wang,2024,Action quality assessment is a more challengin...,yes,The article proposes an action quality assessm...,AQA-7;MTL-AQA
2,10.1109/ITME53901.2021.00048,Skeleton Based Action Quality Assessment of Fi...,H. -Y. Li; Q. Lei; H. -B. Zhang; J. -X. Du,2021,Action quality assessment(AQA) aims at achievi...,yes,The article proposes a skeleton-based AQA meth...,MIT
3,10.1109/ICIP42928.2021.9506257,Action Quality Assessment With Ignoring Scene ...,T. Nagai; S. Takeda; M. Matsumura; S. Shimizu;...,2021,We propose an action quality assessment (AQA) ...,yes,The article proposes an AQA method evaluated o...,Diving
4,10.1109/ICASSP48485.2024.10447069,Multi-Stage Contrastive Regression for Action ...,Q. An; M. Qi; H. Ma,2024,"In recent years, there has been growing intere...",yes,The article proposes Multi-stage Contrastive R...,AQA
...,...,...,...,...,...,...,...,...
91,10.1007/s00521-023-09068-w,Auto-encoding score distribution regression fo...,"Zhang, Boyu; Chen, Jiayuan; Xu, Yinfei; Zhang,...",2023,Assessing the quality of actions in videos is ...,yes,The article proposes a distribution auto-encod...,AQA-7;MTL-AQA;JIGSAWS
92,10.1038/s41597-022-01188-7,Functional movement screen dataset collected w...,"Xing, Qing-Jun; Shen, Yuan-Yuan; Cao, Run; Zon...",2022,This paper presents a dataset for vision-based...,yes,Meets all inclusion criteria: (a) presents a d...,Functional Movement Screen
93,10.3390/en15062071,Manual Operation Evaluation Based on Vectorize...,"He, Fangqiuzi; Liu, Yong; Zhan, Weiwen; Xu, Qi...",2022,The standard of manual operation in smart grid...,yes,Meets all inclusion criteria: (a) uses multi-s...,NaN
94,10.1155/2022/8950732,A Data Mining-Based Model for Evaluating Tenni...,"Chen, Hang",2022,This paper uses data mining technology to math...,yes,Meets all inclusion criteria: (a) uses depth i...,NaN


### Total of 96 papers identified against 101 of the original study

In [12]:
df_validation_reviewed = (
    df_validation[
        (df_validation["Data Source"].notna()) &
        (df_validation["Data Source"].astype(str).str.strip() != "")
    ]
    .reset_index(drop=True)
)

df_validation_reviewed = df_validation_reviewed[["DOI", "Title", "Authors", "Year", "Data Source"]]

dataset_map = {
    "AQA": ["AQA-7"],
    "MIT-Skating": ["MIT-Skate"],
    "MIT Figure Skating": ["MIT-Skate"],
    "MIT-Diving": ["MIT-Dive"],
    "Olympics Diving": ["MIT-Dive"],
    "MIT Olympic Scoring": ["MIT-Skate", "MIT-Dive"],
    "Olympic Sports": ["MIT-Skate", "MIT-Dive"],
    "UNLV-Skating": ["UNLV-Skate"],
    "UNLV-Diving": ["UNLV-Dive"],
    "Gymnastic vaults": ["UNLV-Vault"],
    "UNLV Olympic Scoring": ["UNLV-Skate", "UNLV-Dive", "UNLV-Vault"],
    "FineDiving-HM": ["FineDiving"],
    "Fis-V": ["FIS-V"],
    "Fitness Action": ["Fitness-AQA"],
    "IntelliRehabDS": ["IRDS"],
    "JIGSAWSS": ["JIGSAWS"],
    "MSR Action 3D": ["MSR-Action3D"],
    "MSRC12": ["MSRC-12"],
    "Multitask-AQA": ["MTL-AQA"],
    "Play Therapy 13 (PT13)": ["PT13"]
}


def map_partial_sources(cell):
    if pd.isna(cell):
        return cell

    parts = [p.strip() for p in str(cell).split(";") if p.strip()]
    out = []

    for p in parts:
        matched = False
        for k, v in dataset_map.items():
            pattern = fr"(^|;){re.escape(k)}($|;)"
            if re.fullmatch(k, p):
                out.extend(v)
                matched = True
                break
            elif re.search(pattern, ";" + p + ";"):
                out.extend(v)
                matched = True
                break
        if not matched:
            out.append(p)

    seen = set()
    result = []
    for x in out:
        key = x.strip()
        if key not in seen:
            seen.add(key)
            result.append(key)

    result = [x for x in result if len(x) <= 20]

    return ";".join(result)

df_validation_reviewed["Data Source"] = df_validation_reviewed["Data Source"].apply(map_partial_sources)

df_validation_reviewed["Data Source"] = df_validation_reviewed["Data Source"].replace("", pd.NA)
df_validation_reviewed["Data Source"] = df_validation_reviewed["Data Source"].fillna("self-created")

df_validation_reviewed.to_csv(result_path + "review.csv", index=False)
df_validation_reviewed

,DOI,Title,Authors,Year,Data Source
0,10.1109/ACCESS.2024.3423462,MMW-AQA: Multimodal In-the-Wild Dataset for Ac...,T. Nagai; S. Takeda; S. Suzuki; H. Seshimo,2024,MMW-AQA
1,10.1109/ICAACE61206.2024.10548995,Action Quality Assessment with Multi-scale Tem...,W. Wang; H. Wang; Y. Hao; Q. Wang,2024,AQA-7;MTL-AQA
2,10.1109/ITME53901.2021.00048,Skeleton Based Action Quality Assessment of Fi...,H. -Y. Li; Q. Lei; H. -B. Zhang; J. -X. Du,2021,MIT
3,10.1109/ICIP42928.2021.9506257,Action Quality Assessment With Ignoring Scene ...,T. Nagai; S. Takeda; M. Matsumura; S. Shimizu;...,2021,Diving
4,10.1109/ICASSP48485.2024.10447069,Multi-Stage Contrastive Regression for Action ...,Q. An; M. Qi; H. Ma,2024,AQA-7
...,...,...,...,...,...
66,10.1016/j.patcog.2017.12.007,"Motion analysis: Action detection, recognition...","Patrona, F.; Chatzitofis, A.; Zarpalas, D.; Da...",2018,MSR-Action3D;MSRC-12
67,10.1109/SSCI.2017.8285270,The open online repository of karate motion ca...,"Hachaj, T.; Ogiela, M.R.; Piekarczyk, M.",2017,self-created
68,10.1007/978-3-319-10599-4_36,Assessing the quality of actions,"Pirsiavash, H.; Vondrick, C.; Torralba, A.",2014,MIT-Skate;MIT-Dive
69,10.1007/s00521-023-09068-w,Auto-encoding score distribution regression fo...,"Zhang, Boyu; Chen, Jiayuan; Xu, Yinfei; Zhang,...",2023,AQA-7;MTL-AQA;JIGSAWS


In [13]:
verified_dois_study = set(study_papers_list.keys())
verified_dois_validation = set(df_validation["DOI"].tolist())

matches = verified_dois_study & verified_dois_validation
num_matches = len(matches)

only_study = verified_dois_study - verified_dois_validation
num_study = len(only_study)

only_validation = verified_dois_validation - verified_dois_study
num_validation = len(only_validation)

print("Perfect Matches:", num_matches)
print("Only in Study:", num_study)
print("Only in Validation:", num_validation)

Perfect Matches: 69
Only in Study: 25
Only in Validation: 27


In [14]:
df_final = df_validation[df_validation["Data Source"].isna()].reset_index(drop=True)
df_final

,DOI,Title,Authors,Year,Abstract,Include,Reason,Data Source
0,10.1109/EIECC60864.2023.10456639,A Temporal Action Evaluation Algorithm for Med...,Z. Wang; J. Ruan; J. Zhang; J. Yue,2023,The introduction of temporal action detection ...,maybe,The paper proposes FEBRFormer for temporal act...,NaN
1,10.1109/CVPR52688.2022.00323,Likert Scoring with Grade Decoupling for Long-...,A. Xu; L. -A. Zeng; W. -S. Zheng,2022,Long-term action quality assessment is a task ...,yes,The paper proposes Grade-decoupling Likert Tra...,NaN
2,10.1109/ACCESS.2023.3305372,Contrastive Learning for Action Assessment Usi...,C. -I. Joung; S. Byun; S. Baek,2023,A fine-grained detection of posture problems f...,yes,Meets all inclusion criteria: (a) mentions dat...,NaN
3,10.1109/TVCG.2023.3247092,A Video-Based Augmented Reality System for Hum...,K. Zhou; R. Cai; Y. Ma; Q. Tan; X. Wang; J. Li...,2023,As the most common idiopathic inflammatory myo...,yes,Meets all inclusion criteria: (a) uses JDM dat...,NaN
4,10.1109/MMSP53017.2021.9733638,Piano Skills Assessment,P. Parmar; J. Reddy; B. Morris,2021,Can a computer determine a piano player’s skil...,yes,Meets all inclusion criteria: (a) collects and...,NaN
5,10.1109/TMM.2022.3222681,View-Invariant Center-of-Pressure Metrics Esti...,C. Du; S. Graham; C. Depp; T. Nguyen,2023,"Center of pressure (CoP) metrics, including Co...",yes,Meets all inclusion criteria: (a) releases a n...,NaN
6,10.1109/CVPRW.2017.16,Learning to Score Olympic Events,P. Parmar; B. T. Morris,2017,"Estimating action quality, the process of assi...",yes,Related to computer science and vision-based m...,NaN
7,10.1109/ACCESS.2023.3316009,AI Trainer: Autoencoder Based Approach for Squ...,M. Chariar; S. Rao; A. Irani; S. Suresh; C. S....,2023,Artificial intelligence and computer vision ha...,yes,Related to computer science and vision-based m...,NaN
8,10.1109/ICMEW46912.2020.9106049,Efficient Fitness Action Analysis Based on Spa...,J. Li; H. Cui; T. Guo; Q. Hu; Y. Shen,2020,Human action analysis has been an active resea...,yes,Related to computer science and vision-based m...,NaN
9,10.1109/WACV48630.2021.00044,EAGLE-Eye: Extreme-pose Action Grader using de...,M. Nekoui; F. O. Tito Cruz; L. Cheng,2021,Measuring the quality of a sports action entai...,yes,Related to computer science and vision-based m...,NaN


In [15]:
df_final.to_csv(result_path + "df_evaluated_validation_abstract.csv", index=False)

In [16]:
unique_sources = df_validation['Data Source'].dropna().unique().tolist()
unique_sources = [source.strip() for item in unique_sources for source in item.split(';')]
unique_sources = set(unique_sources)

filtered_sources = set()

for source in unique_sources:
    mapped = dataset_map.get(source, [source])
    filtered_sources.update([s for s in mapped if len(s) <= 20])

filtered_sources.discard("self-created")

print(f"The total number of normalized sources in Validation is: {len(filtered_sources)}\n")
sorted(filtered_sources, key=len)

The total number of normalized sources in Validation is: 45



['KTH',
 'EHE',
 'MIT',
 'RFSJ',
 'RDSD',
 'NETS',
 'PT13',
 'TAQR',
 'LOGO',
 'PD4T',
 'PaSk',
 'IRDS',
 'FIS-V',
 'FR-FS',
 'AQA-7',
 'TASD-2',
 'KIMORE',
 'Diving',
 'Autism',
 'FineFS',
 'UI-PRMD',
 'JIGSAWS',
 'MSRC-12',
 'FineGym',
 'MMW-AQA',
 'MTL-AQA',
 'Pull-Ups',
 'MIT-Dive',
 'OlympicFS',
 'FineRehab',
 'UNLV-Dive',
 'MIT-Skate',
 'TaiChi-24',
 'Fitness-28',
 'UNLV-Vault',
 'FineDiving',
 'UNLV-Skate',
 'Fitness-AQA',
 'AGF-Olympics',
 'Walking Gait',
 'UMONS-TAICHI',
 'MSR-Action3D',
 'Origami Video',
 'TikTok Dance Video',
 'Rhythmic Gymnastics']

In [17]:
verified_datasets_study = set()

for value in study_papers_list.values():
    if value:
        parts = [v.strip() for v in value.split(';') if v.strip()]
        verified_datasets_study.update(parts)

verified_datasets_study.discard("self-created")

print(f"The total number of unique datasets in study_papers_list: {len(verified_datasets_study)}\n")
sorted(verified_datasets_study, key=len)

The total number of unique datasets in study_papers_list: 47



['JDM',
 'EHE',
 'RFSJ',
 'NETS',
 'PT13',
 'LOGO',
 'PD4T',
 'BEST',
 'PISA',
 'PaSk',
 'IRDS',
 'DREAM',
 'MMASD',
 'FIS-V',
 'AQA-7',
 'Squat',
 'TASD-2',
 'FS1OOO',
 'KIMORE',
 'FineFS',
 'UI-PRMD',
 'JIGSAWS',
 'FineGym',
 'Fri2023',
 'MMW-AQA',
 'MTL-AQA',
 'MIT-Dive',
 'OlympicFS',
 'TGC20ReId',
 'CPR-Coach',
 'UNLV-Dive',
 'MIT-Skate',
 'TaiChi-24',
 'Fitness-28',
 'UNLV-Vault',
 'FineDiving',
 'UNLV-Skate',
 'EPIC-Skills',
 'Walking Gait',
 'UMONS-TAICHI',
 'SkatingVerse',
 'SU-41 Gesture',
 'Origami Video',
 'AI-Hub Fitness',
 'Perturbed Workout',
 'Rhythmic Gymnastics',
 'Stroke Rehabilitation']

In [18]:
# Elements in common
common_elements = filtered_sources & verified_datasets_study

# Elements unique to each list
only_in_filtered = filtered_sources - verified_datasets_study
only_in_original = verified_datasets_study - filtered_sources


print("Elements in common:", common_elements)
print("Number of elements in common:", len(common_elements))

print("\nElements in Validation but not in Study:", only_in_filtered)
print("Number:", len(only_in_filtered))

print("\nElements in Study but not in Validation:", only_in_original)
print("Number:", len(only_in_original))

Elements in common: {'Rhythmic Gymnastics', 'RFSJ', 'UI-PRMD', 'TASD-2', 'OlympicFS', 'NETS', 'Walking Gait', 'FIS-V', 'PT13', 'Fitness-28', 'MIT-Dive', 'JIGSAWS', 'UMONS-TAICHI', 'LOGO', 'KIMORE', 'PD4T', 'FineGym', 'EHE', 'AQA-7', 'FineFS', 'PaSk', 'UNLV-Dive', 'MIT-Skate', 'TaiChi-24', 'IRDS', 'MMW-AQA', 'UNLV-Vault', 'MTL-AQA', 'FineDiving', 'Origami Video', 'UNLV-Skate'}
Number of elements in common: 31

Elements in Validation but not in Study: {'TAQR', 'KTH', 'RDSD', 'FineRehab', 'Pull-Ups', 'AGF-Olympics', 'TikTok Dance Video', 'Fitness-AQA', 'MSR-Action3D', 'Autism', 'MIT', 'FR-FS', 'Diving', 'MSRC-12'}
Number: 14

Elements in Study but not in Validation: {'PISA', 'Squat', 'Fri2023', 'FS1OOO', 'Perturbed Workout', 'SkatingVerse', 'SU-41 Gesture', 'DREAM', 'TGC20ReId', 'MMASD', 'EPIC-Skills', 'JDM', 'Stroke Rehabilitation', 'AI-Hub Fitness', 'CPR-Coach', 'BEST'}
Number: 16
